# 02 · Mallas y panel celda-mes
Continúa del notebook `01_exploracion`: parte de `data/interim/accidents_clean.parquet`. La lógica está en `src/mallas.py` y `src/panel.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

INTERIM = ROOT / "data" / "interim"
pd.set_option("display.max_columns", None)

## Mallas
La lógica está en `src/mallas.py`. Las celdas salen **solo del contorno** del municipio (sin mirar accidentes) y entra toda celda que solape con Madrid, aunque lo cubra en parte. Cada resolución se guarda aparte en GeoParquet.

In [ ]:
from mallas import crear_malla, malla_distritos, guardar_mallas

malla_1km = crear_malla(1000)
malla_2km = crear_malla(2000)
malla_dist = malla_distritos()
pd.DataFrame({"celdas": [len(malla_1km), len(malla_2km), len(malla_dist)]}, index=["1 km", "2 km", "distritos"])

### ¿Cuánto cae dentro de Madrid? (celdas de borde)

In [ ]:
resumen = pd.DataFrame({
    "1 km": malla_1km["frac_dentro"].describe(),
    "2 km": malla_2km["frac_dentro"].describe(),
}).round(2)
resumen.loc["celdas < 50 % dentro"] = [(malla_1km["frac_dentro"] < .5).sum(), (malla_2km["frac_dentro"] < .5).sum()]
resumen

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from geografia import TERMINO

fig, ax = plt.subplots(figsize=(7, 7))
malla_1km.plot(ax=ax, column="frac_dentro", cmap="Blues", edgecolor="grey", linewidth=0.3, legend=True)
gpd.read_file(TERMINO).boundary.plot(ax=ax, color="red", linewidth=1)
ax.set_title("Malla de 1 km (color = fracción dentro de Madrid)")
ax.set_axis_off()

### Comprobación con los accidentes
El cruce geométrico estricto deja sin celda a los accidentes que caen **justo sobre una línea** de la malla (las coordenadas son metros enteros, así que X o Y terminadas en `000` coinciden con el borde). Hay que fijar una regla de desempate en la asignación.

In [ ]:
limpios = pd.read_parquet(INTERIM / "accidents_clean.parquet")
sobre_linea = (limpios["coordenada_x_utm"] % 1000 == 0) | (limpios["coordenada_y_utm"] % 1000 == 0)
print("accidentes exactamente sobre una línea de la malla de 1 km:", int(sobre_linea.sum()))
print("accidentes de borde (fuera del límite):", int((~limpios["dentro_municipio"]).sum()))

### Guardado
Una malla por resolución en `data/interim/` (GeoParquet, con geometría). Se regeneran con este notebook.

In [ ]:
rutas = guardar_mallas()
rutas

### Asignación de cada accidente a su celda
Regla de desempate: **intervalo semiabierto** (`x_min <= x < x_min + lado`, igual en Y). Un accidente justo sobre una línea va a la celda de su este/norte, así que nunca queda sin celda ni en dos. Los que caen fuera de todas las celdas (borde del municipio) van a la **más cercana**.

In [ ]:
from mallas import asignar_celda_cuadrada, asignar_distrito

a1 = asignar_celda_cuadrada(limpios, malla_1km)
a2 = asignar_celda_cuadrada(limpios, malla_2km)
ad = asignar_distrito(limpios, malla_dist)

pd.DataFrame(
    {"asignados": [len(a1), len(a2), len(ad)], "por cercanía": [a1["por_cercania"].sum(), a2["por_cercania"].sum(), ad["por_cercania"].sum()],
     "celdas distintas": [a1["cell_id"].nunique(), a2["cell_id"].nunique(), ad["cell_id"].nunique()]},
    index=["1 km", "2 km", "distritos"],
)

In [ ]:
asignacion = pd.DataFrame({
    "num_expediente": limpios["num_expediente"],
    "fecha": limpios["fecha"],
    "mes": limpios["fecha"].dt.to_period("M").dt.to_timestamp(),
    "cell_1km": a1["cell_id"],
    "cell_2km": a2["cell_id"],
    "cell_distrito": ad["cell_id"],
    "por_cercania": a1["por_cercania"] | a2["por_cercania"] | ad["por_cercania"],
})

assert len(asignacion) == len(limpios) and asignacion["num_expediente"].is_unique
assert asignacion[["cell_1km", "cell_2km", "cell_distrito"]].notna().all().all()
assert asignacion["cell_1km"].isin(malla_1km["cell_id"]).all() and asignacion["cell_2km"].isin(malla_2km["cell_id"]).all()
print("OK: todos los accidentes tienen celda en las tres resoluciones; total", len(asignacion))

### Guardado
`data/interim/accidents_celdas.parquet`: una fila por accidente con su celda en cada resolución. Es la entrada del panel celda-mes.

In [ ]:
asignacion.to_parquet(INTERIM / "accidents_celdas.parquet", index=False)
asignacion.head()

## Panel celda × mes
La lógica está en `src/panel.py`. Una fila por cada combinación de celda y mes (84 meses, enero 2019 – diciembre 2025), con `y` = nº de accidentes, **incluidos los ceros**. Tabla pura, sin geometría. Se rellena con 0 en todos los meses porque los siete ficheros cubren cada año completo.

In [ ]:
from panel import PROCESSED, construir_panel

paneles = {
    "1km": construir_panel(asignacion, malla_1km["cell_id"], "cell_1km"),
    "2km": construir_panel(asignacion, malla_2km["cell_id"], "cell_2km"),
    "distritos": construir_panel(asignacion, malla_dist["cell_id"], "cell_distrito"),
}
pd.DataFrame(
    {
        "celdas": {k: p["cell_id"].nunique() for k, p in paneles.items()},
        "filas": {k: len(p) for k, p in paneles.items()},
        "accidentes (suma de y)": {k: int(p["y"].sum()) for k, p in paneles.items()},
        "% de ceros": {k: round(100 * (p["y"] == 0).mean(), 1) for k, p in paneles.items()},
        "y máximo": {k: int(p["y"].max()) for k, p in paneles.items()},
    }
)

In [ ]:
for k, p in paneles.items():
    assert p["y"].sum() == len(asignacion), k
    assert len(p) == p["cell_id"].nunique() * 84, k
    assert not p.duplicated(["cell_id", "target_month"]).any(), k
assert "geometry" not in paneles["1km"].columns
print("OK: la suma de cada panel = accidentes asignados; sin duplicados; sin geometría")

### Accidentes por mes (todo Madrid)
Se ve la caída de abril de 2020 (confinamiento por COVID-19).

In [ ]:
total_mes = paneles["distritos"].groupby("target_month")["y"].sum()
total_mes.plot(figsize=(9, 3), title="Accidentes por mes en Madrid", ylabel="accidentes")

### Celdas de 1 km que nunca tienen un accidente
Son ceros estructurales (periferia, parques, zonas sin calles). Se mantienen en el panel; luego se decide si se filtran.

In [ ]:
por_celda = paneles["1km"].groupby("cell_id")["y"].sum()
nunca = por_celda[por_celda == 0].index
print(len(nunca), "de", len(por_celda), "celdas sin accidentes en los 84 meses")
malla_1km[malla_1km["cell_id"].isin(nunca)]["frac_dentro"].describe().round(2)

### Guardado
Un panel por resolución en `data/processed/`.

In [ ]:
for k, p in paneles.items():
    p.to_parquet(PROCESSED / f"panel_{k}.parquet", index=False)
[f.name for f in sorted(PROCESSED.glob("panel_*.parquet"))]